# K-Means 기반 가중치 양자화 실습 (Deep Compression)

파이토치로 MNIST를 MLP로 간단히 학습시킨 뒤, 학습된 가중치를 **K-Means**로 양자화해 봅니다.

1. 가중치를 safetensors 파일로 저장/로드
2. 가중치 분포 히스토그램
3. **K-Means 양자화** (scikit-learn) — 비슷한 값을 대표값으로 묶기 (비균등)
4. 비트 수에 따른 양자화 오차·정확도

Colab에서는 위에서부터 순서대로 실행하면 됩니다. (런타임 → 모두 실행)

## 0. 준비

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

try:
    from safetensors.torch import save_file, load_file
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "safetensors"])
    from safetensors.torch import save_file, load_file

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. MNIST를 MLP로 학습

작은 3층 MLP(784 → 256 → 128 → 10)를 2 에폭만 학습합니다. 목적은 최고 성능이 아니라 **양자화해 볼 가중치**를 얻는 것이므로 짧게 돌립니다.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(m):
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (m(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(2):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        F.cross_entropy(model(x), y).backward()
        opt.step()
    print(f"epoch {epoch+1}  test acc = {accuracy(model):.4f}")

base_acc = accuracy(model)
print("baseline (float32) accuracy:", base_acc)

# 학습된 가중치를 safetensors 파일로 저장
save_file({k: v.cpu().contiguous() for k, v in model.state_dict().items()},
          "mnist_mlp.safetensors")
print("saved -> mnist_mlp.safetensors")

## 2. 가중치 파일 저장하고 다시 불러오기 (safetensors)

임의로 만든 행렬이 아니라, 방금 **파일로 저장한 실제 체크포인트**(`mnist_mlp.safetensors`)를 다시 읽어 다뤄봅니다.

`.pth`(pickle 기반) 대신 **safetensors** 포맷을 씁니다.

- **안전**: pickle이 아니라 임의 코드 실행 위험이 없음
- **빠름**: zero-copy 로딩, 필요한 텐서만 골라 읽기 가능
- **표준**: HuggingFace 모델 가중치의 사실상 표준 포맷

> 이미 가진 `.safetensors` 파일이 있다면 Colab 파일 탭에 업로드한 뒤 경로만 바꿔서 그대로 실습할 수 있습니다.

In [ ]:
import os
print("file size:", os.path.getsize("mnist_mlp.safetensors"), "bytes")

# safetensors 파일에서 state_dict 로드 (텐서는 CPU로 반환됨)
state = load_file("mnist_mlp.safetensors")
print("\nstate_dict keys / shapes:")
for k, v in state.items():
    print(f"  {k:14s} {tuple(v.shape)}  ({v.numel()} params)")

# 새 모델에 로드해서 그대로 사용
model = MLP().to(device)
model.load_state_dict(load_file("mnist_mlp.safetensors"))
model.eval()
print("\nreloaded model test acc:", accuracy(model))

## 3. 가중치 분포 히스토그램

파일에서 불러온 첫 번째 층(`fc1`)의 가중치를 펼쳐 히스토그램을 그려봅니다. 신경망 가중치의 전형적인 특징 — **0 근처에 종 모양으로 몰려 있고 꼬리가 긴** 분포 — 를 확인할 수 있습니다.

In [ ]:
w = model.fc1.weight.detach().cpu().numpy()
print("fc1.weight shape:", w.shape, " (총", w.size, "개 파라미터)")
print(f"min={w.min():.3f}  max={w.max():.3f}  mean={w.mean():.4f}  std={w.std():.3f}")

plt.figure(figsize=(7, 3.5))
plt.hist(w.reshape(-1), bins=200, color="#3b82f6")
plt.title("fc1 weight distribution (float32)")
plt.xlabel("weight value"); plt.ylabel("count")
plt.tight_layout(); plt.show()

## 4. K-Means 양자화

비슷한 값의 가중치를 $k = 2^N$개의 클러스터로 묶고, 각 클러스터의 중심값(centroid)을 **코드북**으로 저장합니다. 각 가중치는 자기가 속한 클러스터의 대표값으로 복원됩니다. 클러스터링은 **scikit-learn의 `KMeans`** 를 사용합니다.

- **비균등(non-uniform)**: 값이 몰린 곳(0 근처)에 대표값이 촘촘하게 배치됨
- 초기값은 min~max를 균등 분할한 선형 초기화 (지역 최적해 회피)

빠른 실행을 위해 코드북은 가중치의 부분표본으로 학습(fit)한 뒤 전체에 적용합니다.

In [ ]:
def kmeans_quantize(w, n_bits, fit_sample=20000):
    """scikit-learn KMeans로 가중치를 양자화. 복원값, 코드북, 인덱스를 반환."""
    shape = w.shape
    flat = w.reshape(-1, 1)
    k = 2 ** n_bits
    lo, hi = float(flat.min()), float(flat.max())
    init = np.linspace(lo, hi, k).reshape(-1, 1)          # 선형 초기화
    n = len(flat)
    idx = np.random.choice(n, min(fit_sample, n), replace=False)
    km = KMeans(n_clusters=k, init=init, n_init=1, max_iter=50).fit(flat[idx])
    codebook = km.cluster_centers_.reshape(-1)
    indices = km.predict(flat)
    recon = codebook[indices].reshape(shape)
    return recon, codebook, indices

N_BITS = 2
recon_km, codebook, indices = kmeans_quantize(w, N_BITS)
mse_km = ((w - recon_km) ** 2).mean()

num = w.size; k = 2 ** N_BITS
orig_bits = 32 * num
comp_bits = N_BITS * num + 32 * k     # 인덱스 + 코드북
print(f"[K-Means] {N_BITS}-bit, k={k} clusters")
print("codebook (centroids):", np.sort(codebook).round(3))
print(f"quantization MSE = {mse_km:.6f}")
print(f"compression: {orig_bits/comp_bits:.2f}x  ({orig_bits//8} B -> {comp_bits//8} B)")

plt.figure(figsize=(7, 3.5))
plt.hist(w.reshape(-1), bins=200, alpha=0.4, color="#3b82f6", label="original")
plt.hist(recon_km.reshape(-1), bins=200, color="#e0a020", label="K-Means quantized")
for c in np.sort(codebook):
    plt.axvline(c, color="#e0a020", lw=1, alpha=0.7)
plt.title(f"K-Means quantization ({N_BITS}-bit, {k} levels)")
plt.xlabel("weight value"); plt.ylabel("count"); plt.legend()
plt.tight_layout(); plt.show()

## 5. 비트 수에 따른 정확도

모델의 모든 Linear 층 가중치를 K-Means로 양자화한 뒤(복원값으로 교체) MNIST 테스트 정확도가 어떻게 변하는지 비트 수를 바꿔가며 봅니다. (양자화 후 복원해 float로 추론하는 fake quantization 방식)

In [ ]:
import copy

def eval_kmeans(model, n_bits):
    m = copy.deepcopy(model)
    with torch.no_grad():
        for layer in [m.fc1, m.fc2, m.fc3]:
            wl = layer.weight.detach().cpu().numpy()
            recon, *_ = kmeans_quantize(wl, n_bits)
            layer.weight.copy_(torch.tensor(recon, dtype=torch.float32, device=device))
    return accuracy(m)

bits_list = [2, 3, 4, 8]
acc_km = [eval_kmeans(model, b) for b in bits_list]

print(f"{'bits':>5} | {'K-Means acc':>12}")
for b, a in zip(bits_list, acc_km):
    print(f"{b:>5} | {a:>12.4f}")
print(f"{'float':>5} | {base_acc:>12.4f}")

plt.figure(figsize=(6.5, 4))
plt.axhline(base_acc, color="gray", ls=":", label=f"float32 ({base_acc:.3f})")
plt.plot(bits_list, acc_km, "o-", color="#e0a020", label="K-Means")
plt.title("MNIST test accuracy vs bit-width (K-Means)")
plt.xlabel("bits per weight"); plt.ylabel("test accuracy")
plt.xticks(bits_list); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 마무리 & 더 해보기

- K-Means는 대표값을 **데이터 분포에 맞춰(비균등)** 배치하므로, 낮은 비트에서 균등 양자화보다 오차가 작은 경향이 있습니다.
- 다만 추론 시 코드북 디코드가 필요해 **저장 용량만** 줄어듭니다 (연산은 여전히 float).

**더 해볼 것**
1. 양자화 후 코드북을 fine-tuning해 정확도 되살리기
2. Conv 층이 있는 모델(LeNet 등)로 확장
3. 인덱스를 허프만 코딩으로 추가 압축